# Problem 2 — TEST multiclass relabeling
Rebuilds correct multiclass labels for the 290 test videos (150 Normal + 140 anomalous),
using the category folder name (fixing the `Testing_Normal_Videos_Anomaly` mislabel) and
real temporal annotations for clip-level labels. **Does not require re-running VideoMAE**
on videos you've already extracted — only recomputes labels and joins them to your
existing `features/test/*.npz` files by `(video_filename, clip_id)`. Missing videos (if any)
are extracted at the end, resumably.

Independent of the train/val notebook — run both in parallel.

In [ ]:
# ============================================================
# Configuration -- IDENTICAL in both notebooks on purpose.
# These values match what already produced your existing test/*.npz files
# (4s clips, 2s stride / 50% overlap, 16 frames @ 4fps, CLS-token pooling,
# MCG-NJU/videomae-base-finetuned-kinetics). Do not change independently
# between the two notebooks or clip_id alignment breaks.
# ============================================================
from pathlib import Path
import os, re, json, math, random, shutil, hashlib, warnings
import numpy as np
import pandas as pd
import cv2

from sklearn.model_selection import train_test_split

SEED = 42
random.seed(SEED); np.random.seed(SEED)

NUM_FRAMES   = 16
SAMPLE_FPS   = 4.0
CLIP_DURATION = NUM_FRAMES / SAMPLE_FPS   # 4.0 seconds
CLIP_STRIDE   = CLIP_DURATION / 2         # 2.0 seconds, 50% overlap
ANNOTATION_FPS = 30.0
OVERLAP_THRESHOLD = 0.50

MODEL_NAME = "MCG-NJU/videomae-base-finetuned-kinetics"

CATEGORIES = ["Normal","Abuse","Arrest","Arson","Assault","Burglary","Explosion",
              "Fighting","RoadAccidents","Robbery","Shooting","Shoplifting",
              "Stealing","Vandalism"]
CAT2IDX = {c: i for i, c in enumerate(CATEGORIES)}
IDX2CAT = {i: c for c, i in CAT2IDX.items()}

INPUT_ROOT = Path("/kaggle/input")
WORK_ROOT  = Path("/kaggle/working/ucf_videomae")
SPLIT_ROOT    = WORK_ROOT / "splits"
FEATURE_ROOT  = WORK_ROOT / "features"
METADATA_ROOT = WORK_ROOT / "metadata"
for p in [SPLIT_ROOT, FEATURE_ROOT, METADATA_ROOT]:
    p.mkdir(parents=True, exist_ok=True)
for split in ["train", "val", "test"]:
    (FEATURE_ROOT / split).mkdir(parents=True, exist_ok=True)

VIDEO_EXTENSIONS = {".mp4", ".avi", ".mov", ".mkv"}
print("Config loaded. Categories:", CATEGORIES)


In [ ]:
# ============================================================
# Locate the UCF-Crime dataset root (same auto-discovery as the reference notebook)
# ============================================================
def find_ucf_root(base_dir=INPUT_ROOT):
    candidates = []
    for p in base_dir.iterdir():
        if not p.is_dir():
            continue
        has_anomaly_train = any(x.name.lower() == "anomaly_train.txt" for x in p.rglob("*") if x.is_file())
        has_temporal = any("temporal_anomaly_annotation" in x.name.lower() for x in p.rglob("*") if x.is_file())
        if has_anomaly_train and has_temporal:
            candidates.append(p)
    if not candidates:
        raise FileNotFoundError("Could not automatically find the UCF-Crime dataset under /kaggle/input.")
    return candidates[0]

DATASET_ROOT = find_ucf_root()
print("DATASET_ROOT:", DATASET_ROOT)
for item in sorted(DATASET_ROOT.iterdir()):
    print(("DIR " if item.is_dir() else "FILE"), item.name)


In [ ]:
# ============================================================
# Path-resolution helpers (identical to the reference notebook)
# ============================================================
def find_files_by_name(root, filename):
    return [p for p in root.rglob("*") if p.is_file() and p.name.lower() == filename.lower()]

def find_dirs_by_name(root, names):
    target = {n.lower() for n in names}
    return [p for p in root.rglob("*") if p.is_dir() and p.name.lower() in target]

all_video_paths = [p for p in DATASET_ROOT.rglob("*") if p.is_file() and p.suffix.lower() in VIDEO_EXTENSIONS]
print(f"Total video files discovered: {len(all_video_paths):,}")

by_filename, by_relpath = {}, {}
for p in all_video_paths:
    by_filename.setdefault(p.name.lower(), []).append(p)
    rel = str(p.relative_to(DATASET_ROOT)).replace("\\", "/").lower()
    by_relpath[rel] = p

def normalize_split_line(line):
    line = line.strip().replace("\\", "/")
    if not line or line.startswith("#"):
        return None
    return line.strip()

def resolve_split_entry(entry):
    entry = normalize_split_line(entry)
    if entry is None:
        return None
    entry_low = entry.lower().lstrip("./")
    matches = [p for rel, p in by_relpath.items() if rel.endswith(entry_low)]
    if matches:
        return matches[0]
    filename = Path(entry).name.lower()
    candidates = by_filename.get(filename, [])
    if len(candidates) == 1:
        return candidates[0]
    if len(candidates) > 1:
        entry_parent = Path(entry).parent.name.lower()
        for p in candidates:
            if p.parent.name.lower() == entry_parent:
                return p
        return candidates[0]
    return None

def read_split_file(path):
    entries, unresolved = [], []
    with open(path, "r", errors="ignore") as fh:
        for line in fh:
            e = normalize_split_line(line)
            if e is None:
                continue
            resolved = resolve_split_entry(e)
            if resolved is not None:
                entries.append(resolved)
            else:
                unresolved.append(e)
    return entries, unresolved

def category_from_path(path):
    return path.parent.name

def video_id(video_path):
    # Hash the relative path so duplicate filenames in different folders never collide.
    # MUST stay identical across both notebooks -- this is what makes feature filenames
    # match up with whatever previously extracted the test set.
    rel = str(Path(video_path).relative_to(DATASET_ROOT))
    return hashlib.md5(rel.encode("utf-8")).hexdigest()[:16]


## Build the official TEST set (anomaly + normal) with correct multiclass labels

In [ ]:
anomaly_test_file = find_files_by_name(DATASET_ROOT, "Anomaly_Test.txt")
assert anomaly_test_file, "Anomaly_Test.txt not found"
ANOMALY_TEST_FILE = anomaly_test_file[0]
print("Anomaly_Test.txt:", ANOMALY_TEST_FILE)

anomaly_test_paths, anomaly_test_unresolved = read_split_file(ANOMALY_TEST_FILE)
print("Official anomaly TEST videos resolved:", len(anomaly_test_paths), "(expect ~140)")
if anomaly_test_unresolved:
    print("Unresolved entries:", anomaly_test_unresolved[:10])

# The genuinely-normal test videos live here -- NOT anomalous, despite the folder name.
normal_test_dirs = find_dirs_by_name(DATASET_ROOT, ["Testing_Normal_Videos", "Testing_Normal_Videos_Anomaly"])
normal_test_paths = sorted(set(p for d in normal_test_dirs for p in d.rglob("*")
                                if p.is_file() and p.suffix.lower() in VIDEO_EXTENSIONS))
print("Normal TEST videos:", len(normal_test_paths), "(expect ~150)")


In [ ]:
anomaly_test_df = pd.DataFrame({
    "video_path": [str(p) for p in anomaly_test_paths],
    "video_filename": [p.name for p in anomaly_test_paths],
    "category": [category_from_path(p) for p in anomaly_test_paths],
})
normal_test_df = pd.DataFrame({
    "video_path": [str(p) for p in normal_test_paths],
    "video_filename": [p.name for p in normal_test_paths],
    # Force "Normal" explicitly -- this is THE fix for the mislabeling bug.
    # Do not use the literal folder name (Testing_Normal_Videos_Anomaly) as the category.
    "category": ["Normal"] * len(normal_test_paths),
})

test_df = pd.concat([anomaly_test_df, normal_test_df], ignore_index=True)
test_df = test_df.drop_duplicates(subset="video_filename").reset_index(drop=True)
test_df["video_label_idx"] = test_df["category"].map(CAT2IDX)
test_df["split"] = "test"

print("Total TEST videos:", len(test_df), "(expect 290)")
print(test_df["category"].value_counts())
assert test_df["video_label_idx"].isna().sum() == 0, "Unmapped category found -- check CATEGORIES list."


## Parse temporal annotations and assign clip-level multiclass labels

In [ ]:
temporal_files = [p for p in DATASET_ROOT.rglob("*") if p.is_file()
                  and "temporal_anomaly_annotation" in p.name.lower()]
assert temporal_files, "Temporal annotation file not found"
TEMPORAL_FILE = temporal_files[0]
print("Temporal annotation file:", TEMPORAL_FILE)

def parse_temporal_annotations(annotation_file):
    rows = []
    with open(annotation_file, "r", encoding="utf-8", errors="ignore") as f:
        for raw in f:
            parts = raw.strip().split()
            if len(parts) < 4:
                continue
            filename, category = parts[0], parts[1]
            try:
                start1, end1 = float(parts[2]), float(parts[3])
            except ValueError:
                continue
            start2 = float(parts[4]) if len(parts) >= 5 else -1
            end2   = float(parts[5]) if len(parts) >= 6 else -1
            intervals = [(s, e) for s, e in [(start1, end1), (start2, end2)] if s >= 0 and e >= 0]
            for s_frame, e_frame in intervals:
                rows.append({"video_filename": filename, "category": category,
                             "start_sec": s_frame / ANNOTATION_FPS, "end_sec": e_frame / ANNOTATION_FPS})
    return pd.DataFrame(rows)

temporal_df = parse_temporal_annotations(TEMPORAL_FILE)
test_anomaly_filenames = set(anomaly_test_df["video_filename"].str.lower())
temporal_test_df = temporal_df[temporal_df["video_filename"].str.lower().isin(test_anomaly_filenames)].copy()

missing_annotations = sorted(test_anomaly_filenames - set(temporal_test_df["video_filename"].str.lower()))
print("Anomaly test videos with temporal annotation:", temporal_test_df["video_filename"].nunique())
print("Anomaly test videos MISSING temporal annotation:", len(missing_annotations))
if missing_annotations:
    print(missing_annotations[:20])

annotation_map = {}
for _, r in temporal_test_df.iterrows():
    annotation_map.setdefault(r["video_filename"].lower(), []).append((float(r["start_sec"]), float(r["end_sec"])))


In [ ]:
def interval_overlap(a_start, a_end, b_start, b_end):
    overlap = max(0.0, min(a_end, b_end) - max(a_start, b_start))
    return overlap / max(a_end - a_start, 1e-9)

def assign_test_clip_label(video_filename, start_sec, end_sec, category, video_label_idx):
    # Normal video: every clip is Normal, always.
    if category == "Normal":
        return CAT2IDX["Normal"], "Normal", 0.0

    intervals = annotation_map.get(video_filename.lower(), [])
    if not intervals:
        # Anomalous video with no matched annotation: keep the weak video-level
        # label as a fallback, flagged via label_source so it's excluded from
        # frame-level ROC-AUC later (same caveat as the binary version).
        return video_label_idx, category, np.nan

    max_overlap = max(interval_overlap(start_sec, end_sec, s, e) for s, e in intervals)
    if max_overlap >= OVERLAP_THRESHOLD:
        return video_label_idx, category, max_overlap
    else:
        # This specific clip doesn't show the anomaly -- it's normal footage
        # even though it's cut from an anomalous video.
        return CAT2IDX["Normal"], "Normal", max_overlap


In [ ]:
def get_video_info(video_path):
    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        raise RuntimeError(f"Could not open video: {video_path}")
    fps = cap.get(cv2.CAP_PROP_FPS)
    frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    width, height = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)), int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    cap.release()
    duration = frame_count / fps if fps and fps > 0 else None
    return {"fps": fps, "frame_count": frame_count, "width": width, "height": height, "duration_sec": duration}

def generate_clip_windows(row):
    info = get_video_info(row["video_path"])
    duration = info["duration_sec"]
    if duration is None or duration < CLIP_DURATION:
        return []
    clips, start, clip_id = [], 0.0, 0
    while start + CLIP_DURATION <= duration + 1e-6:
        end = start + CLIP_DURATION
        clips.append({"video_path": row["video_path"], "video_filename": row["video_filename"],
                       "category": row["category"], "video_label_idx": int(row["video_label_idx"]),
                       "split": "test", "clip_id": clip_id,
                       "start_sec": round(start, 4), "end_sec": round(end, 4),
                       "video_fps": info["fps"], "video_duration_sec": duration,
                       "width": info["width"], "height": info["height"]})
        clip_id += 1
        start += CLIP_STRIDE
    return clips

records = []
for i, row in test_df.iterrows():
    for c in generate_clip_windows(row):
        label_idx, label_name, overlap = assign_test_clip_label(
            c["video_filename"], c["start_sec"], c["end_sec"], c["category"], c["video_label_idx"])
        c["clip_label_idx"] = label_idx
        c["clip_label_name"] = label_name
        c["annotation_overlap"] = overlap
        c["label_source"] = ("normal_video" if c["category"] == "Normal"
                              else "temporal_annotation" if not np.isnan(overlap)
                              else "video_level_fallback_no_annotation")
        records.append(c)
    if (i + 1) % 50 == 0:
        print(f"  processed {i+1}/{len(test_df)} videos")

test_clips = pd.DataFrame(records)
print("\nTest clips:", len(test_clips))
print(test_clips["clip_label_name"].value_counts())
print("\nLabel source breakdown:")
print(test_clips["label_source"].value_counts())


In [ ]:
test_clips.to_parquet(METADATA_ROOT / "test_clips_multiclass.parquet", index=False)
print("Saved corrected multiclass test clip metadata.")
print("\nSanity check -- Normal-category videos should have ZERO non-Normal clip labels:")
normal_rows = test_clips[test_clips["category"] == "Normal"]
print("  non-Normal labels among Normal videos:", (normal_rows["clip_label_name"] != "Normal").sum(), "(must be 0)")


## Join to existing features (no re-extraction needed for videos already processed)

In [ ]:
# ============================================================
# Resume support: restore any previously-extracted features for this split
# if you attached an earlier output as a Kaggle input dataset.
# Safe to run even if nothing is found -- just prints 0 copied.
# ============================================================
def restore_previous_features(split, search_name="ucf_videomae/features"):
    candidates = list(Path("/kaggle/input").rglob(f"{search_name}/{split}"))
    dest = FEATURE_ROOT / split
    dest.mkdir(parents=True, exist_ok=True)
    copied = 0
    for cand in candidates:
        for src in cand.glob("*.npz"):
            dst = dest / src.name
            if not dst.exists():
                shutil.copy2(src, dst)
                copied += 1
    print(f"[{split}] restored {copied} previously-extracted feature files "
          f"(total now on disk: {len(list(dest.glob(\'*.npz\')))})")

restore_previous_features("test")


In [ ]:
def video_id(video_path):
    rel = str(Path(video_path).relative_to(DATASET_ROOT))
    return hashlib.md5(rel.encode("utf-8")).hexdigest()[:16]

test_df["video_id"] = test_df["video_path"].apply(video_id)
existing_ids = {f.stem for f in (FEATURE_ROOT / "test").glob("*.npz")}
missing_df = test_df[~test_df["video_id"].isin(existing_ids)]
print(f"Test videos already extracted: {len(test_df) - len(missing_df)} / {len(test_df)}")
print(f"Test videos MISSING features: {len(missing_df)}")
if len(missing_df):
    print(missing_df[["video_filename", "category"]].head(10))


The next cell loads VideoMAE. It's only actually *used* if `missing_df` is non-empty — harmless to run either way, but you can skip it if the previous cell showed 0 missing.

In [ ]:
# ============================================================
# VideoMAE loading + feature extraction (identical scheme to what produced
# your existing features -- CLS-token pooling from the last hidden state)
# ============================================================
import torch
from transformers import VideoMAEImageProcessor, VideoMAEForVideoClassification

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)

processor = VideoMAEImageProcessor.from_pretrained(MODEL_NAME)
model = VideoMAEForVideoClassification.from_pretrained(MODEL_NAME).to(DEVICE).eval()
print("Model loaded. hidden_size:", model.config.hidden_size)

def read_clip_frames(video_path, start_sec, end_sec, num_frames=NUM_FRAMES, sample_fps=SAMPLE_FPS):
    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        raise RuntimeError(f"Could not open: {video_path}")
    video_fps = cap.get(cv2.CAP_PROP_FPS)
    if video_fps <= 0:
        cap.release()
        raise RuntimeError(f"Invalid FPS for: {video_path}")
    timestamps = start_sec + np.arange(num_frames) / sample_fps
    frames = []
    for t in timestamps:
        frame_idx = int(round(t * video_fps))
        cap.set(cv2.CAP_PROP_POS_FRAMES, frame_idx)
        ok, frame = cap.read()
        if not ok:
            cap.release()
            raise RuntimeError(f"Could not read frame {frame_idx} at {t:.3f}s from {video_path}")
        frames.append(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
    cap.release()
    if len(frames) != num_frames:
        raise RuntimeError(f"Expected {num_frames} frames, got {len(frames)}")
    return frames

def pool_videomae_output(outputs):
    hidden = outputs.hidden_states[-1]
    return hidden[:, 0, :] if hidden.ndim == 3 else hidden  # CLS token

def extract_one_clip_feature(video_path, start_sec, end_sec):
    frames = read_clip_frames(video_path, start_sec, end_sec)
    inputs = processor(frames, return_tensors="pt")
    inputs = {k: v.to(DEVICE) for k, v in inputs.items()}
    with torch.no_grad():
        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=(DEVICE.type == "cuda")):
            outputs = model(**inputs, output_hidden_states=True, return_dict=True)
    return pool_videomae_output(outputs).squeeze(0).float().cpu().numpy()

def extract_video_features(video_clip_df, split):
    assert len(video_clip_df) > 0
    video_path = video_clip_df.iloc[0]["video_path"]
    vid = video_id(video_path)
    out_file = FEATURE_ROOT / split / f"{vid}.npz"
    if out_file.exists():
        return "skipped", out_file

    feature_list, clip_ids, starts, ends = [], [], [], []
    for _, row in video_clip_df.iterrows():
        try:
            feat = extract_one_clip_feature(row["video_path"], float(row["start_sec"]), float(row["end_sec"]))
            feature_list.append(feat)
            clip_ids.append(int(row["clip_id"]))
            starts.append(float(row["start_sec"]))
            ends.append(float(row["end_sec"]))
        except Exception as e:
            print(f"ERROR: {row['video_filename']} clip {row['clip_id']}: {e}")
            raise

    features = np.stack(feature_list).astype(np.float32)
    np.savez(out_file, features=features, clip_id=np.array(clip_ids),
              start_sec=np.array(starts), end_sec=np.array(ends), video_id=vid)
    return "saved", out_file

def group_clips_by_video(clip_df):
    return [group for _, group in clip_df.groupby("video_filename", sort=False)]

def run_split_extraction(clip_df, split):
    groups = group_clips_by_video(clip_df)
    saved = skipped = 0
    for i, group in enumerate(groups, 1):
        status, _ = extract_video_features(group, split)
        saved += status == "saved"
        skipped += status == "skipped"
        if i % 10 == 0 or i == len(groups):
            print(f"{split}: {i}/{len(groups)} videos | saved={saved}, skipped={skipped}")
    return saved, skipped


In [ ]:
if len(missing_df) > 0:
    missing_clips = test_clips[test_clips["video_filename"].isin(missing_df["video_filename"])]
    saved, skipped = run_split_extraction(missing_clips, "test")
    print(f"Extraction complete for gap videos: saved={saved}, skipped={skipped}")
else:
    print("Nothing missing -- skipping extraction, reusing existing features/test/*.npz as-is.")


## Build the final joined manifest (features on disk + corrected multiclass labels)

In [ ]:
manifest_rows = []
for _, row in test_df.iterrows():
    npz_path = FEATURE_ROOT / "test" / f"{row['video_id']}.npz"
    if not npz_path.exists():
        print(f"[WARNING] still missing after extraction: {row['video_filename']}")
        continue
    data = np.load(npz_path)
    clip_ids_in_file = data["clip_id"]
    clips_for_video = test_clips[test_clips["video_filename"] == row["video_filename"]].set_index("clip_id")
    for cid in clip_ids_in_file:
        cid = int(cid)
        if cid not in clips_for_video.index:
            continue  # shouldn't happen if clip windowing matches, but don't crash if it does
        meta = clips_for_video.loc[cid]
        manifest_rows.append({
            "video_filename": row["video_filename"], "video_id": row["video_id"],
            "category": meta["category"], "clip_id": cid,
            "start_sec": meta["start_sec"], "end_sec": meta["end_sec"],
            "clip_label_idx": meta["clip_label_idx"], "clip_label_name": meta["clip_label_name"],
            "annotation_overlap": meta["annotation_overlap"], "label_source": meta["label_source"],
            "feature_npz": str(npz_path), "feature_row": int(np.where(clip_ids_in_file == cid)[0][0]),
        })

final_manifest = pd.DataFrame(manifest_rows)
final_manifest.to_parquet(METADATA_ROOT / "test_features_multiclass_manifest.parquet", index=False)
print("Final test manifest rows:", len(final_manifest))
print(final_manifest["clip_label_name"].value_counts())
print("\nSaved -> ", METADATA_ROOT / "test_features_multiclass_manifest.parquet")
print("Load a feature vector with: np.load(row.feature_npz)[\'features\'][row.feature_row]")
